In [1]:
suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(batchelor))
suppressPackageStartupMessages(library(argparse))

here::i_am("mapping/run/mnn/mapping_mnn.R")

# Load mapping functions
source(here::here("mapping/run/mnn/mapping_functions.R"))

# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/03_Stat3_RNA/code



In [2]:
# I/O
io$path2atlas <- io$atlas.basedir
io$path2query <- io$basedir

# START TEST ##
args = list()
args$atlas_stages <- c("E7.5","E7.75")
args$query_samples <- opts$samples
args$query_sce <- io$rna.sce
args$query_sce <- paste0(io$basedir,"/processed/SingleCellExperiment.rds")
args$atlas_sce <- io$rna.atlas.sce
args$query_metadata <- paste0(io$basedir,"/results/rna/doublet_detection/sample_metadata_after_doublets.txt.gz")
args$atlas_metadata <- io$rna.atlas.metadata
args$test <- TRUE
args$npcs <- 5
args$n_neighbours <- 25
args$use_marker_genes <- FALSE
args$cosine_normalisation <- FALSE
args$outdir <- paste0(io$basedir,"/results/rna/mapping/test")
# END TEST ##

In [3]:
if (isTRUE(args$test)) print("Test mode activated...")


[1] "Test mode activated..."


In [4]:
meta_query <- fread(args$query_metadata) %>% 
  .[pass_rnaQC==TRUE & doublet_call==FALSE & sample%in%args$query_samples]

In [5]:
head(meta_query)

cell,sample,barcode,nFeature_RNA,nCount_RNA,mitochondrial_percent_RNA,ribosomal_percent_RNA,stage,tdTom,idx,tdTom_corr,pass_rnaQC,doublet_score,doublet_call
<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>,<lgl>,<int>,<lgl>,<lgl>,<dbl>,<lgl>
SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1,SLX-21143_SITTA2_HTJH3DSX2,AAACCCAAGATGTTCC-1,4837,53199,1.47,22.87,E8.5,TRUE,1,TRUE,TRUE,0.09,FALSE
SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1,SLX-21143_SITTA2_HTJH3DSX2,AAACCCATCAGACCTA-1,4973,24165,0.95,19.26,E8.5,TRUE,3,TRUE,TRUE,0.24,FALSE
SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1,SLX-21143_SITTA2_HTJH3DSX2,AAACGAACAATGTTGC-1,5585,34341,0.22,26.76,E8.5,TRUE,5,TRUE,TRUE,0.22,FALSE
SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1,SLX-21143_SITTA2_HTJH3DSX2,AAACGAACACACCAGC-1,5016,41496,1.66,25.01,E8.5,TRUE,6,TRUE,TRUE,0.11,FALSE
SLX-21143_SITTA2_HTJH3DSX2#AAACGAAGTGATAGTA-1,SLX-21143_SITTA2_HTJH3DSX2,AAACGAAGTGATAGTA-1,4308,44249,0.60,24.26,E8.5,TRUE,7,TRUE,TRUE,0.10,FALSE
SLX-21143_SITTA2_HTJH3DSX2#AAACGAAGTTAAGACA-1,SLX-21143_SITTA2_HTJH3DSX2,AAACGAAGTTAAGACA-1,3826,17237,1.42,28.43,E8.5,TRUE,8,TRUE,TRUE,0.10,FALSE


In [6]:
################
## Load query ##
################

# Load cell metadata
meta_query <- fread(args$query_metadata) %>% 
  .[pass_rnaQC==TRUE & doublet_call==FALSE & sample%in%args$query_samples]
if (isTRUE(args$test)) meta_query <- head(meta_query,n=1000)

# Load SingleCellExperiment
sce_query <- load_SingleCellExperiment(args$query_sce, cells = meta_query$cell, remove_non_expressed_genes = TRUE)

# Update colData
tmp <- meta_query %>% .[cell%in%colnames(sce_query)] %>% setkey(cell) %>% .[colnames(sce_query)]
stopifnot(tmp$cell == colnames(sce_query))
colData(sce_query) <- tmp %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_query),] %>% DataFrame()

################
## Load atlas ##
################

# Load cell metadata
meta_atlas <- fread(args$atlas_metadata) %>%
  .[stage%in%args$atlas_stages] %>%
  .[,sample:=factor(sample)]

# Filter
if (isTRUE(args$test)) meta_atlas <- head(meta_atlas,n=1000)

# Load SingleCellExperiment
sce_atlas <- load_SingleCellExperiment(args$atlas_sce, normalise = TRUE, cells = meta_atlas$cell, remove_non_expressed_genes = TRUE)

# Update colData
tmp <- meta_atlas %>% .[cell%in%colnames(sce_atlas)] %>% setkey(cell) %>% .[colnames(sce_atlas)]
stopifnot(tmp$cell == colnames(sce_atlas))
colData(sce_atlas) <- tmp %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_atlas),] %>% DataFrame()

# Sanity cehcks
stopifnot(sum(is.na(rownames(sce_atlas)))==0)
stopifnot(sum(duplicated(rownames(sce_atlas)))==0)


In [7]:
#####################
## Define gene set ##
#####################

# Get gene metadata
gene_metadata <- fread(io$gene_metadata) %>% .[,c("chr","ens_id","symbol")] %>%
  .[symbol!="" & ens_id%in%rownames(sce_atlas)] %>%
  .[!duplicated(symbol)]

# Intersect genes
genes.intersect <- intersect(rownames(sce_query), rownames(sce_atlas))

# Filter some genes manually
genes.intersect <- genes.intersect[grep("^Rik|Rik$|^mt-|^Rps|^Rpl|^Gm",genes.intersect,invert=T)]
genes.intersect <- genes.intersect[!genes.intersect=="Xist"]
genes.intersect <- genes.intersect[!genes.intersect%in%gene_metadata[chr=="chrY",symbol]]


In [8]:
head(genes.intersect, 1000)

[1] "Sox17"      "Mrpl15"     "Lypla1"     "Tcea1"      "Atp6v1h"   
   [6] "Rb1cc1"     "Pcmtd1"     "Rrs1"       "Vcpip1"     "Sgk3"      
  [11] "Snhg6"      "Cops5"      "Cspp1"      "Arfgef1"    "Prex2"     
  [16] "Sulf1"      "Ncoa2"      "Tram1"      "Lactb2"     "Terf1"     
  [21] "Rdh10"      "Stau2"      "Ube2w"      "Tmem70"     "Jph1"      
  [26] "Crispld1"   "Mcm3"       "Tram2"      "Tmem14a"    "Ogfrl1"    
  [31] "Smap1"      "Sdhaf4"     "Fam135a"    "Col9a1"     "Lmbrd1"    
  [36] "Phf3"       "Ptp4a1"     "Prim2"      "Rab23"      "Bag2"      
  [41] "Zfp451"     "Dst"        "Ccdc115"    "Imp4"       "Ptpn18"    
  [46] "Cfc1"       "Fam168b"    "Plekhb2"    "Hs6st1"     "Uggt1"     
  [51] "Kansl3"     "Lman2l"     "Cnnm4"      "Cnnm3"      "Ankrd39"   
  [56] "Sema4c"     "Fam178b"    "Cox5b"      "Actr1b"     "Tmem131"   
  [61] "Coa5"       "Unc50"      "Mgat4a"     "Tsga10"     "Lipt1"     
  [66] "Mitd1"      "Mrpl30"     "Txndc9"     "Eif5b"      "Rev1"      
  [71] "Aff3"       "Pdcl3"      "Cnot11"     "Rnf149"     "Map4k4"    
  [76] "Mrps9"      "Tgfbrap1"   "AI597479"   "Fhl2"       "Uxs1"      
  [81] "Tpp2"       "Tex30"      "Kdelc1"     "Bivm"       "Ercc5"     
  [86] "Gulp1"      "Wdr75"      "Slc40a1"    "Dnah7b"     "Slc39a10"  
  [91] "Nabp1"      "Myo1b"      "Stat1"      "Gls"        "Nab1"      
  [96] "Nemp2"      "Inpp1"      "Hibch"      "Pms1"       "Ormdl1"    
 [101] "Osgepl1"    "Asnsd1"     "Stk17b"     "Gtf3c3"     "Pgap1"     
 [106] "Sf3b1"      "Coq10b"     "Hspd1"      "Hspe1"      "Mob4"      
 [111] "Mars2"      "Satb2"      "Tyw5"       "Spats2l"    "Kctd18"    
 [116] "Sgol2a"     "Bzw1"       "Clk1"       "Ppil3"      "Nif3l1"    
 [121] "Orc2"       "Fam126b"    "Ndufb3"     "Cflar"      "Casp8"     
 [126] "Trak2"      "Stradb"     "Tmem237"    "Als2"       "Fzd7"      
 [131] "Sumo1"      "Nop58"      "Bmpr2"      "Fam117b"    "Wdr12"     
 [136] "Carf"       "Nbeal1"     "Cyp20a1"    "Abi2"       "Raph1"     
 [141] "Nrp2"       "Ino80d"     "Ndufs1"     "Eef1b2"     "Zdbf2"     
 [146] "Fastkd2"    "Klf7"       "Creb1"      "Mettl21a"   "Ccnyl1"    
 [151] "Fzd5"       "Plekhm3"    "Idh1"       "Pikfyve"    "Map2"      
 [156] "Rpe"        "Kansl1l"    "Acadl"      "Lancl1"     "Ikzf2"     
 [161] "Bard1"      "Atic"       "Fn1"        "Mreg"       "Pecr"      
 [166] "Xrcc5"      "Smarcal1"   "Igfbp2"     "Igfbp5"     "Tns1"      
 [171] "Arpc2"      "Aamp"       "Pnkd"       "Tmbim1"     "Ctdsp1"    
 [176] "Usp37"      "Cnot9"      "Zfp142"     "Bcs1l"      "Rnf25"     
 [181] "Ttll4"      "Cyp27a1"    "Wnt6"       "Ihh"        "Nhej1"     
 [186] "Cnppd1"     "Fam134a"    "Zfand2b"    "Abcb6"      "Atg9a"     
 [191] "Ankzf1"     "Stk16"      "Tuba4a"     "Dnajb2"     "Ptprn"     
 [196] "Dnpep"      "Des"        "Speg"       "Chpf"       "Obsl1"     
 [201] "Stk11ip"    "Slc4a3"     "Epha4"      "Sgpp2"      "Farsb"     
 [206] "Acsl3"      "Utp14b"     "Ap1s3"      "Wdfy1"      "Mrpl44"    
 [211] "Serpine2"   "Cul3"       "Irs1"       "Rhbdd1"     "Mff"       
 [216] "Agfg1"      "Pid1"       "Trip12"     "Fbxo36"     "Cab39"     
 [221] "Itm2c"      "Psmd1"      "Armc9"      "B3gnt7"     "Ncl"       
 [226] "Ptma"       "Pde6d"      "Cops7b"     "Dis3l2"     "Eif4e2"    
 [231] "Gigyf2"     "Atg16l1"    "Sag"        "Dgkd"       "Usp40"     
 [236] "Dnajb3"     "Hjurp"      "Arl4c"      "Sh3bp4"     "Agap1"     
 [241] "Gbx2"       "Ackr3"      "Cops8"      "Lrrfip1"    "Ube2f"     
 [246] "Scly"       "Ilkap"      "Hes6"       "Traf3ip1"   "Asb1"      
 [251] "Twist2"     "Hdac4"      "Ndufa10"    "Gpc1"       "Dusp28"    
 [256] "Rnpepl1"    "Capn10"     "Kif1a"      "Mterf4"     "Pask"      
 [261] "Ppp1r7"     "Hdlbp"      "Stk25"      "Bok"        "Thap4"     
 [266] "Atg4b"      "Dtymk"      "Ing5"       "D2hgdh"     "Fam174a"   
 [271] "D1Ertd622e" "Ppip5k2"    "Gin1"       "Pam"        "Pign"      
 [276] "Zcchc2"     "Phlpp1"     "Bcl2"    

In [31]:

# Subset SingleCellExperiment objects
sce_query  <- sce_query[genes.intersect,]
sce_atlas <- sce_atlas[genes.intersect,]

In [45]:
#######################
## Feature selection ##
#######################

if (args$use_marker_genes) {
  # Load marker genes
  marker_genes.dt <- fread(io$rna.atlas.marker_genes)
  genes_to_use <- genes.intersect[genes.intersect%in%unique(marker_genes.dt$gene)]
} else {
  # Load gene statistics from the atlas
  gene_stats.dt <- fread(paste0(io$atlas.basedir,"/results/gene_statistics/gene_statistics.txt.gz")) %>%
    .[gene%in%genes.intersect]
  genes_to_use <- gene_stats.dt %>% setorder(-var_pseudobulk, na.last = T) %>% head(n=2500) %>% .$gene  
  
  # Calculate mean-variance relationship and extract HVGs
  # decomp <- modelGeneVar(sce_atlas, block=sce_atlas$sample)
  # genes_to_use <- rownames(decomp)[decomp$p.value<=0.01 & decomp$mean>0.1]
}

stopifnot(genes_to_use%in%rownames(sce_atlas))
stopifnot(genes_to_use%in%rownames(sce_query))

In [47]:
head(genes_to_use, 20)

[1] "Hbb-y"    "Hbb-bh1"  "Ptma"     "Hba-a1"   "Npm1"     "Fth1"    
 [7] "Ttr"      "Ftl1"     "Hsp90ab1" "Rbp4"     "Apoa1"    "Ppia"    
[13] "Apoe"     "Eef1a1"   "Tpt1"     "Ldha"     "H2afz"    "Hspa8"   
[19] "Ubb"      "Actg1"

# MapWrap unpacked

In [48]:
  sce_atlas = sce_atlas
  meta_atlas = meta_atlas
  sce_query = sce_query
  meta_query = meta_query
  genes = genes_to_use
  npcs = args$npcs
  k = args$n_neighbours
  cosineNorm = args$cosine_normalisation
  order = NULL

In [49]:
   
  # Normalisation
  message(sprintf("Normalizing joint dataset using cosineNorm=%s...",cosineNorm))
  sce_all <- joint.normalisation(sce_query, sce_atlas, cosineNorm)
  message("Done\n")

Normalizing joint dataset using cosineNorm=FALSE...

Done




In [50]:
  
  # Feature selection
  if (is.null(genes)) {
    message("Genes not provided. Computing highly variable genes...")
    # hvgs <- getHVGs(sce_all, block=c(meta_atlas$sample, meta_query$sample))
    genes <- getHVGs(sce_all, block=sce_all$block)
    message("Done\n")
  } else {
    message(sprintf("%d Genes provided...",length(genes)))
  }

2500 Genes provided...



In [51]:
  # Dimensionality reduction
  message("Performing PCA...")
  big_pca <- multiBatchPCA(
    sce_all,
    batch = sce_all$block,
    subset.row = genes,
    d = npcs,
    preserve.single = TRUE,
    assay.type = if (cosineNorm) "cosineNorm" else "logcounts"
  )[[1]]
  rownames(big_pca) <- colnames(sce_all) 
  atlas_pca <- big_pca[1:ncol(sce_atlas),]
  query_pca   <- big_pca[-(1:ncol(sce_atlas)),]
  message("Done\n")

Performing PCA...

Done




In [52]:
  
  # Batch effect correction for the atlas
  message("Batch effect correction for the atlas...")  
  order_df        <- meta_atlas[!duplicated(meta_atlas$sample), c("stage", "sample")]
  order_df$ncells <- sapply(order_df$sample, function(x) sum(meta_atlas$sample == x))
  order_df$stage  <- factor(order_df$stage, levels = rev(c("E9.5",
                                       "E9.25",
                                       "E9.0",
                                       "E8.75",
                                       "E8.5",
                                       "E8.25",
                                       "E8.0",
                                       "E7.75",
                                       "E7.5",
                                       "E7.25",
                                       "mixed_gastrulation",
                                       "E7.0",
                                       "E6.75",
                                       "E6.5")))
  order_df       <- order_df[order(order_df$stage, order_df$ncells, decreasing = TRUE),]
  order_df$stage <- as.character(order_df$stage)
  
  set.seed(42)
  pca_atlas_corrected <- doBatchCorrect(counts         = logcounts(sce_atlas[genes,]), 
                                    timepoints      = meta_atlas$stage, 
                                    samples         = meta_atlas$sample, 
                                    timepoint_order = order_df$stage, 
                                    sample_order    = order_df$sample, 
                                    pc_override     = atlas_pca,
                                    npc             = npcs)
  message("Done\n")

Batch effect correction for the atlas...

Loading required package: BiocParallel

Done




In [54]:
  # Mapping query to batch-corrected atlas
  message("MNN mapping...")              
  # correct <- reducedMNN(rbind(pca_atlas_corrected, query_pca), batch = sce_all$block)[["corrected"]]
  correct <- reducedMNN(rbind(pca_atlas_corrected, query_pca),
                      # batch=c(rep("ATLAS", dim(meta_atlas)[1]), meta_query$sample),
                      batch = as.character(sce_all$block),
                      merge.order = order)$corrected
  pca_atlas_corrected <- correct[1:nrow(atlas_pca),]
  pca_query_corrected   <- correct[-(1:nrow(atlas_pca)),]

MNN mapping...



In [56]:
head(pca_query_corrected)
head(pca_atlas_corrected)

SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1,-14.987037,2.413490,24.6406411,-3.032402,5.767520
SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1,-7.940671,7.905278,-14.2722171,-16.816031,10.581002
SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1,-8.726419,8.758685,-0.9571918,-8.901286,-1.698947
SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1,-13.366618,3.451749,23.3918650,-5.954716,0.584606
SLX-21143_SITTA2_HTJH3DSX2#AAACGAAGTGATAGTA-1,-15.784547,1.576234,23.8164706,-1.795425,6.121257
SLX-21143_SITTA2_HTJH3DSX2#AAACGAAGTTAAGACA-1,-8.859923,9.763747,2.7037113,-5.096928,-4.358171


cell_361,-22.18608,-5.858202,-7.477659,13.22464068,-8.494032
cell_362,-12.44580,6.015637,2.600329,-1.15762130,2.156080
cell_363,-10.66507,8.345092,7.358897,-3.35566261,3.066436
cell_364,-10.77508,6.928303,9.191929,-4.99766121,-4.773238
cell_365,-12.98190,5.618987,1.879723,0.01229901,2.966236
cell_366,-10.58148,6.606713,6.264774,-6.22517346,-5.805861


In [59]:
meta_atlas$celltype = meta_atlas$celltype_extended_atlas

In [60]:
  mapping <- get_meta(pca_atlas = pca_atlas_corrected,
                      meta_atlas = meta_atlas,
                      pca_query = pca_query_corrected,
                      meta_query = meta_query,
                      k = k)
  message("Done\n")

Done




In [61]:

  

  


  



  
  # Mapping scores
  message("Computing mapping scores...") 
  out <- list()
  for (i in seq(from = 1, to = k)) {
    out$closest.cells[[i]]     <- sapply(mapping, function(x) x$cells.mapped[i])
    out$celltypes.mapped[[i]]  <- sapply(mapping, function(x) x$celltypes.mapped[i])
    out$cellstages.mapped[[i]] <- sapply(mapping, function(x) x$stages.mapped[i])
  }  
  multinomial.prob <- getMappingScore(out)
  message("Done\n")
  
  # Prepare output
  message("Writing output...") 
  out$pca_atlas_corrected <- pca_atlas_corrected
  out$pca_query_corrected <- pca_query_corrected
  ct <- sapply(mapping, function(x) x$celltype.mapped); is.na(ct) <- lengths(ct) == 0
  st <- sapply(mapping, function(x) x$stage.mapped); is.na(st) <- lengths(st) == 0
  cm <- sapply(mapping, function(x) x$cells.mapped[1]); is.na(cm) <- lengths(cm) == 0
  out$mapping <- data.frame(
      cell            = names(mapping), 
      celltype.mapped = unlist(ct),
      stage.mapped    = unlist(st),
      closest.cell    = unlist(cm))
  
  out$mapping <- cbind(out$mapping,multinomial.prob)
  out$pca <- big_pca
  message("Done\n")
  
  return(out)


Computing mapping scores...

Done


Writing output...

Done




In [62]:
str(out)

List of 7
 $ closest.cells      :List of 25
  ..$ : Named chr [1:1000] "cell_393" "cell_965" "cell_1108" "cell_393" ...
  .. ..- attr(*, "names")= chr [1:1000] "SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1" ...
  ..$ : Named chr [1:1000] "cell_592" "cell_711" "cell_1386" "cell_1434" ...
  .. ..- attr(*, "names")= chr [1:1000] "SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1" ...
  ..$ : Named chr [1:1000] "cell_398" "cell_1099" "cell_1196" "cell_2671" ...
  .. ..- attr(*, "names")= chr [1:1000] "SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1" ...
  ..$ : Named chr